In [ ]:
import sys
import os
# Add project root to path to import utils
sys.path.append(os.path.abspath('..'))

import torch
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from torch_geometric.loader import DataLoader
from utils import get_dataset
# Try importing LitGraphNN, handling potential location differences

from utils.litmodels import LitGraphNN


# 1. Configuration
task = 'diam'  # Choose task: sssp, ecc, diam, charge, energy
config = {
    'conv_layer': 'GCNConv',
    'num_layers': 8,
    'hidden_dim': 128,
    'lr': 0.001,
    'weight_decay': 1e-5,
    'batch_size': 512,
    'gnn_type': 'GNN', # Maps to GNN in litmodels.py
    'dropout_prob': 0.5,
    'activ_fun': 'relu',
}

# 2. Load Dataset
print(f"Loading dataset for task: {task}")
data_train, data_val, data_test, num_feat, num_class = get_dataset(
    root="../data",
    task=task
)

scaling_factor = data_train.scaling_factor.get(task, 1.0)
print(f"Scaling factor: {scaling_factor}")

train_loader = DataLoader(data_train, batch_size=config['batch_size'], shuffle=True, num_workers=4)
val_loader = DataLoader(data_val, batch_size=config['batch_size'], shuffle=False, num_workers=4)
test_loader = DataLoader(data_test, batch_size=config['batch_size'], shuffle=False, num_workers=4)



In [ ]:
# 3. Initialize Model
model = LitGraphNN(
    input_dim=num_feat,
    output_dim=num_class,
    node_level_task=True if task not in ["diam", "energy"] else False,
    scaling_factor=scaling_factor,
    task=task,
    **config
)

In [ ]:
# 4. Setup Trainer with Checkpointing
checkpoint_dir = "../checkpoints/notebook_gcn"
checkpoint_callback = ModelCheckpoint(
    dirpath=checkpoint_dir,
    filename=f"{task}-gcn-{{epoch:02d}}-{{val_loss:.4f}}",
    save_top_k=1,
    monitor="val_loss",
    mode="min"
)

trainer = L.Trainer(
    max_epochs=500, # Adjust as needed
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    callbacks=[checkpoint_callback],
    enable_progress_bar=True,
    log_every_n_steps=10
)

In [ ]:
# 5. Train
print("Starting training...")
trainer.fit(model, train_loader, val_loader)

print(f"Best model saved at: {checkpoint_callback.best_model_path}")

# 6. Test
print("Testing best model...")
trainer.test(model, test_loader, ckpt_path="best")